# Research Paper RAG — Exploration Notebook

**Dr. Hua Li** · Personal research assistant

This notebook walks through the RAG pipeline end to end:
1. Ingest papers → chunk → embed → store in ChromaDB
2. Query with semantic search
3. Generate grounded answers with an LLM
4. Evaluate retrieval quality

Great for a portfolio demo — shows a complete, production-style RAG system built on your own data.

In [ ]:
import sys
sys.path.insert(0, '../src')

from ingest import run_ingestion
from rag import query, retrieve, build_context, get_collection
from sentence_transformers import SentenceTransformer

print('Imports OK')

## Step 1: Ingest papers

In [ ]:
# First-time setup — run with reset=True to rebuild from scratch
run_ingestion(reset=True)

## Step 2: Test retrieval

In [ ]:
embedder   = SentenceTransformer('all-MiniLM-L6-v2')
collection = get_collection()

test_query = 'user interest modeling personalization'
chunks = retrieve(test_query, embedder, collection, top_k=4)

print(f'Retrieved {len(chunks)} chunks for query: "{test_query}"\n')
for i, c in enumerate(chunks, 1):
    meta = c['metadata']
    print(f'{i}. [{c["similarity"]:.3f}] {meta.get("title", "?")[:65]}')
    print(f'   {meta.get("venue", "")} ({meta.get("year", "?")})')
    print(f'   {c["text"][:120]}...')
    print()

## Step 3: Full RAG query

In [ ]:
# Try different questions — notice how retrieval differs
questions = [
    'What work have I done on user modeling and personalization?',
    'How does my research connect to modern RAG systems?',
    'What DARPA and IARPA programs did I contribute to?',
    'Summarize my contributions to information retrieval',
]

question = questions[0]  # change index to try others
answer = query(question)
print(answer)

## Step 4: Retrieval evaluation — similarity score distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

eval_queries = [
    ('user modeling', 'On-topic'),
    ('neural networks deep learning', 'Off-topic'),
    ('adaptive systems personalization', 'On-topic'),
    ('blockchain crypto', 'Off-topic'),
    ('information retrieval knowledge', 'On-topic'),
]

fig, ax = plt.subplots(figsize=(10, 4))
colors = {'On-topic': '#1D9E75', 'Off-topic': '#D85A30'}

for query_text, label in eval_queries:
    chunks = retrieve(query_text, embedder, collection, top_k=5)
    sims   = [c['similarity'] for c in chunks]
    ax.plot(range(1, 6), sims, 'o-', label=f'{query_text} ({label})', color=colors[label], alpha=0.8)

ax.set_xlabel('Rank')
ax.set_ylabel('Cosine Similarity')
ax.set_title('Retrieval Similarity Scores by Query Type')
ax.legend(bbox_to_anchor=(1, 1))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nOn-topic queries should show higher similarity scores — this validates the embedding quality.')

## Step 5: Connecting your past work to modern AI engineering

Your papers map directly to modern RAG concepts:

| Your 2003–2014 work | Modern RAG equivalent |
|---|---|
| Adaptive nearest-neighbor search (KDD 2003) | Embedding-based retrieval / ANN search |
| Model-guided information discovery (CIKM 2005) | Knowledge-graph-augmented RAG |
| Managing analysis context (WIDM 2012) | Conversational memory / session-aware RAG |
| Adaptive interest modeling (MILCOM 2014) | Query rewriting / user-adaptive retrieval |

**Portfolio talking point:** "I've been building retrieval and recommendation systems since KDD 2003 — this RAG project is a direct extension of that lineage with modern LLM infrastructure."